# Synthetic Pore Networks

# Bundle of Parallel Capillaries

In [ ]:
def create_heterogeneous_tight_rock(shape, porosity_target=0.10, voxel_size_um=0.05, seed=None):
    """
    Genera un medio poroso tipo "bundle of tubes" con capilares rectos paralelos.
    
    Características:
    - Capilares rectos (sin tortuosidad) como en modelos clásicos de Poiseuille
    - Distribución regular con variación en diámetros
    - Capilares paralelos que atraviesan el dominio
    - Heterogeneidad controlada en los diámetros
    """
    
    print(f"="*60)
    print(f"GENERANDO MEDIO POROSO TIPO HAZ DE CAPILARES")
    print(f"="*60)
    print(f"Dimensiones: {shape}")
    print(f"Porosidad objetivo: {porosity_target:.1%}")
    print(f"Tamaño de voxel: {voxel_size_um:.3f} μm")
    
    if seed is not None:
        np.random.seed(seed)
    
    # PASO 1: Definir distribución de diámetros de capilares
    # Distribución heterogénea pero organizada
    
    # Escala 1: Capilares grandes (20% del total)
    large_pore_size_um = 0.5  # Diámetro grande
    fraction_large = 0.20
    
    # Escala 2: Capilares medianos (50% del total)
    medium_pore_size_um = 0.3  # Diámetro medio
    fraction_medium = 0.50
    
    # Escala 3: Capilares pequeños (30% del total)
    small_pore_size_um = 0.15  # Diámetro pequeño
    fraction_small = 0.30
    
    # Inicializar medio
    medium = np.zeros(shape, dtype=bool)
    
    # PASO 2: Calcular número y espaciado de capilares
    # Dirección principal de los capilares (eje X por defecto)
    flow_direction = 2  # 0=Z, 1=Y, 2=X
    
    if flow_direction == 2:  # Capilares en dirección X
        cross_section = shape[0] * shape[1]
        length = shape[2]
        n_y, n_z = shape[1], shape[0]
    elif flow_direction == 1:  # Capilares en dirección Y
        cross_section = shape[0] * shape[2]
        length = shape[1]
        n_x, n_z = shape[2], shape[0]
    else:  # Capilares en dirección Z
        cross_section = shape[1] * shape[2]
        length = shape[0]
        n_x, n_y = shape[2], shape[1]
    
    # Estimar número de capilares basado en porosidad objetivo
    avg_radius_um = (large_pore_size_um * fraction_large + 
                     medium_pore_size_um * fraction_medium + 
                     small_pore_size_um * fraction_small) / 2
    avg_radius_vox = avg_radius_um / voxel_size_um
    avg_area_per_capillary = np.pi * avg_radius_vox**2
    
    total_pore_area_needed = cross_section * porosity_target
    n_capillaries = int(total_pore_area_needed / avg_area_per_capillary)
    
    # Distribuir capilares por tamaño
    n_large_pores = max(2, int(n_capillaries * fraction_large))
    n_medium_pores = max(5, int(n_capillaries * fraction_medium))
    n_small_pores = max(5, int(n_capillaries * fraction_small))
    
    print(f"\nDistribución de capilares rectos:")
    print(f"• Grandes (Ø {large_pore_size_um:.2f} μm): {n_large_pores} capilares")
    print(f"• Medianos (Ø {medium_pore_size_um:.2f} μm): {n_medium_pores} capilares")
    print(f"• Pequeños (Ø {small_pore_size_um:.2f} μm): {n_small_pores} capilares")
    print(f"• Total: {n_large_pores + n_medium_pores + n_small_pores} capilares")
    
    # PASO 3: Función para crear capilares rectos
    def create_straight_capillary(position, diameter_um, direction='x'):
        """
        Crea un capilar recto a través del dominio.
        
        Args:
            position: (y, z) para dirección X, (x, z) para Y, o (x, y) para Z
            diameter_um: Diámetro del capilar
            direction: 'x', 'y', o 'z'
        """
        radius_vox = diameter_um / (2 * voxel_size_um)
        
        if direction == 'x':
            y_center, z_center = position
            for x in range(shape[2]):
                for y in range(max(0, int(y_center - radius_vox - 1)), 
                              min(shape[1], int(y_center + radius_vox + 2))):
                    for z in range(max(0, int(z_center - radius_vox - 1)), 
                                  min(shape[0], int(z_center + radius_vox + 2))):
                        if (y - y_center)**2 + (z - z_center)**2 <= radius_vox**2:
                            medium[z, y, x] = True
        
        elif direction == 'y':
            x_center, z_center = position
            for y in range(shape[1]):
                for x in range(max(0, int(x_center - radius_vox - 1)), 
                              min(shape[2], int(x_center + radius_vox + 2))):
                    for z in range(max(0, int(z_center - radius_vox - 1)), 
                                  min(shape[0], int(z_center + radius_vox + 2))):
                        if (x - x_center)**2 + (z - z_center)**2 <= radius_vox**2:
                            medium[z, y, x] = True
        
        else:  # direction == 'z'
            x_center, y_center = position
            for z in range(shape[0]):
                for x in range(max(0, int(x_center - radius_vox - 1)), 
                              min(shape[2], int(x_center + radius_vox + 2))):
                    for y in range(max(0, int(y_center - radius_vox - 1)), 
                                  min(shape[1], int(y_center + radius_vox + 2))):
                        if (x - x_center)**2 + (y - y_center)**2 <= radius_vox**2:
                            medium[z, y, x] = True
    
    # PASO 4: Generar posiciones de capilares con distribución semi-regular
    print(f"\nGenerando capilares rectos paralelos...")
    
    # Lista para almacenar todas las posiciones y tamaños
    capillary_specs = []
    
    # Generar grid semi-regular
    if flow_direction == 2:  # X direction
        # Estimar espaciado basado en número de capilares
        n_grid = int(np.sqrt(n_large_pores + n_medium_pores + n_small_pores))
        dy = shape[1] / (n_grid + 1)
        dz = shape[0] / (n_grid + 1)
        
        # Capilares grandes - distribución más espaciada
        for i in range(n_large_pores):
            # Posición base en grid
            grid_y = (i % int(np.sqrt(n_large_pores)) + 1) * shape[1] / (int(np.sqrt(n_large_pores)) + 2)
            grid_z = (i // int(np.sqrt(n_large_pores)) + 1) * shape[0] / (int(np.sqrt(n_large_pores)) + 2)
            
            # Añadir variación aleatoria
            y_pos = grid_y + np.random.uniform(-dy/4, dy/4)
            z_pos = grid_z + np.random.uniform(-dz/4, dz/4)
            
            # Asegurar margen
            margin = large_pore_size_um / (2 * voxel_size_um) + 1
            y_pos = np.clip(y_pos, margin, shape[1] - margin)
            z_pos = np.clip(z_pos, margin, shape[0] - margin)
            
            capillary_specs.append(((y_pos, z_pos), large_pore_size_um, 'large'))
    
    # Función para verificar solapamiento
    def check_overlap(new_pos, new_radius, existing_specs):
        for (pos, diam, _) in existing_specs:
            dist = np.sqrt((new_pos[0] - pos[0])**2 + (new_pos[1] - pos[1])**2)
            min_dist = (new_radius + diam/2) / voxel_size_um + 1
            if dist < min_dist:
                return True
        return False
    
    # Capilares medianos - llenar espacios
    attempts = 0
    added_medium = 0
    while added_medium < n_medium_pores and attempts < n_medium_pores * 10:
        if flow_direction == 2:
            y_pos = np.random.uniform(0.1, 0.9) * shape[1]
            z_pos = np.random.uniform(0.1, 0.9) * shape[0]
            pos = (y_pos, z_pos)
        
        radius = medium_pore_size_um / 2
        
        if not check_overlap(pos, radius, capillary_specs):
            capillary_specs.append((pos, medium_pore_size_um, 'medium'))
            added_medium += 1
        attempts += 1
    
    # Capilares pequeños - llenar espacios restantes
    attempts = 0
    added_small = 0
    while added_small < n_small_pores and attempts < n_small_pores * 10:
        if flow_direction == 2:
            y_pos = np.random.uniform(0.05, 0.95) * shape[1]
            z_pos = np.random.uniform(0.05, 0.95) * shape[0]
            pos = (y_pos, z_pos)
        
        radius = small_pore_size_um / 2
        
        if not check_overlap(pos, radius, capillary_specs):
            capillary_specs.append((pos, small_pore_size_um, 'small'))
            added_small += 1
        attempts += 1
    
    # PASO 5: Crear todos los capilares
    direction_map = {0: 'z', 1: 'y', 2: 'x'}
    flow_dir = direction_map[flow_direction]
    
    print(f"  Creando {len(capillary_specs)} capilares en dirección {flow_dir.upper()}...")
    
    for i, (position, diameter, size_class) in enumerate(capillary_specs):
        # Añadir pequeña variación en el diámetro para heterogeneidad
        diameter_variation = np.random.uniform(0.9, 1.1)
        actual_diameter = diameter * diameter_variation
        
        create_straight_capillary(position, actual_diameter, flow_dir)
        
        if (i + 1) % 50 == 0:
            print(f"    Progreso: {i + 1}/{len(capillary_specs)} capilares creados")
    
    # PASO 6: Análisis final
    final_porosity = np.sum(medium) / medium.size
    
    # Verificar conectividad
    inlet = np.zeros(shape, dtype=bool)
    if flow_direction == 2:  # X
        inlet[:, :, :5] = True
    elif flow_direction == 1:  # Y
        inlet[:, :5, :] = True
    else:  # Z
        inlet[:5, :, :] = True
    
    connected = ps.filters.trim_disconnected_blobs(im=medium, inlets=inlet)
    connectivity = np.sum(connected) / np.sum(medium) if np.sum(medium) > 0 else 0
    
    # Calcular distribución de diámetros
    dt_final = distance_transform_edt(medium)
    pore_radii = dt_final[medium] * voxel_size_um * 2  # Convertir a diámetros
    heterogeneity = np.std(pore_radii) / np.mean(pore_radii) if len(pore_radii) > 0 else 0
    
    print(f"\nResultados finales:")
    print(f"• Porosidad final: {final_porosity:.3f}")
    print(f"• Conectividad: {connectivity:.3f}")
    print(f"• Heterogeneidad: {heterogeneity:.3f}")
    print(f"• Número total de capilares: {len(capillary_specs)}")
    
    if len(pore_radii) > 0:
        print(f"• Diámetro medio efectivo: {np.mean(pore_radii):.3f} μm")
        print(f"• Rango de diámetros: {np.min(pore_radii):.3f} - {np.max(pore_radii):.3f} μm")
    
    # Tortuosidad = 1.0 para capilares rectos
    print(f"• Tortuosidad: 1.00 (capilares rectos)")
    
    # Información adicional sobre la distribución
    n_large_actual = sum(1 for _, _, t in capillary_specs if t == 'large')
    n_medium_actual = sum(1 for _, _, t in capillary_specs if t == 'medium')
    n_small_actual = sum(1 for _, _, t in capillary_specs if t == 'small')
    
    print(f"\nDistribución final de capilares:")
    print(f"• Grandes: {n_large_actual} ({n_large_actual/len(capillary_specs)*100:.1f}%)")
    print(f"• Medianos: {n_medium_actual} ({n_medium_actual/len(capillary_specs)*100:.1f}%)")
    print(f"• Pequeños: {n_small_actual} ({n_small_actual/len(capillary_specs)*100:.1f}%)")
    
    domain_size_um = [dim * voxel_size_um for dim in shape]
    
    return {
        'medium': medium,
        'shape': shape,
        'voxel_size_um': voxel_size_um,
        'domain_size_um': domain_size_um,
        'porosity_target': porosity_target,
        'porosity_actual': final_porosity,
        'connectivity': connectivity,
        'heterogeneity': heterogeneity,
        'pore_scales': {
            'large': large_pore_size_um,
            'medium': medium_pore_size_um,
            'small': small_pore_size_um
        }
    }

# Multidirectional Channels Network

In [ ]:
# ========== FUNCIONES DE GENERACIÓN DE MEDIO POROSO ==========

def create_heterogeneous_tight_rock(shape, porosity_target=0.10, voxel_size_um=0.05, seed=None):
    """
    Genera un medio poroso heterogéneo optimizado para rocas tight.
    
    Características:
    - Distribución jerárquica de tamaños para garantizar heterogeneidad
    - Canales base para conectividad
    - Tamaños apropiados para la resolución
    """
    
    print(f"="*60)
    print(f"GENERANDO MEDIO POROSO HETEROGÉNEO OPTIMIZADO")
    print(f"="*60)
    print(f"Dimensiones: {shape}")
    print(f"Porosidad objetivo: {porosity_target:.1%}")
    print(f"Tamaño de voxel: {voxel_size_um:.3f} μm")
    
    if seed is not None:
        np.random.seed(seed)
    
    # PASO 1: Definir distribución jerárquica de tamaños
    # Asegurar heterogeneidad con 3 escalas distintas
    
    # Escala 1: Poros grandes (10% del total)
    large_pore_size_um = 0.8  # 16 voxels de radio
    n_large_pores = max(5, int(porosity_target * 0.1 * np.prod(shape) / 1000))
    
    # Escala 2: Poros medianos (30% del total)
    medium_pore_size_um = 0.4  # 8 voxels de radio
    n_medium_pores = max(20, int(porosity_target * 0.3 * np.prod(shape) / 500))
    
    # Escala 3: Poros pequeños (60% del total)
    small_pore_size_um = 0.2  # 4 voxels de radio
    n_small_pores = max(100, int(porosity_target * 0.6 * np.prod(shape) / 200))
    
    print(f"\nDistribución jerárquica de poros:")
    print(f"• Grandes ({large_pore_size_um:.1f} μm): {n_large_pores} poros")
    print(f"• Medianos ({medium_pore_size_um:.1f} μm): {n_medium_pores} poros")
    print(f"• Pequeños ({small_pore_size_um:.1f} μm): {n_small_pores} poros")
    
    # Inicializar medio
    medium = np.zeros(shape, dtype=bool)
    
    # PASO 2: Crear red de canales base para conectividad
    print(f"\nCreando red de canales base...")
    
    # Canales principales en las 3 direcciones
    channel_radius = int(small_pore_size_um / (2 * voxel_size_um))
    
    # Canal X
    y_center, z_center = shape[1]//2, shape[0]//2
    for x in range(shape[2]):
        for y in range(max(0, y_center-channel_radius), min(shape[1], y_center+channel_radius+1)):
            for z in range(max(0, z_center-channel_radius), min(shape[0], z_center+channel_radius+1)):
                if (y-y_center)**2 + (z-z_center)**2 <= channel_radius**2:
                    medium[z, y, x] = True
    
    # Canal Y
    x_center, z_center = shape[2]//2, shape[0]//2
    for y in range(shape[1]):
        for x in range(max(0, x_center-channel_radius), min(shape[2], x_center+channel_radius+1)):
            for z in range(max(0, z_center-channel_radius), min(shape[0], z_center+channel_radius+1)):
                if (x-x_center)**2 + (z-z_center)**2 <= channel_radius**2:
                    medium[z, y, x] = True
    
    # Canal Z
    x_center, y_center = shape[2]//2, shape[1]//2
    for z in range(shape[0]):
        for x in range(max(0, x_center-channel_radius), min(shape[2], x_center+channel_radius+1)):
            for y in range(max(0, y_center-channel_radius), min(shape[1], y_center+channel_radius+1)):
                if (x-x_center)**2 + (y-y_center)**2 <= channel_radius**2:
                    medium[z, y, x] = True
    
    # Canales diagonales para mejor conectividad
    for i in range(0, shape[0], shape[0]//4):
        for j in range(0, shape[1], shape[1]//4):
            start_x = 0 if (i+j)%2 == 0 else shape[2]-1
            end_x = shape[2]-1 if start_x == 0 else 0
            
            # Crear canal diagonal
            for t in np.linspace(0, 1, shape[2]):
                x = int(start_x + t * (end_x - start_x))
                y = int(j + t * shape[1]//4)
                z = int(i + t * shape[0]//4)
                
                if 0 <= y < shape[1] and 0 <= z < shape[0]:
                    for dy in range(-channel_radius, channel_radius+1):
                        for dz in range(-channel_radius, channel_radius+1):
                            if dy**2 + dz**2 <= channel_radius**2:
                                yy, zz = y+dy, z+dz
                                if 0 <= yy < shape[1] and 0 <= zz < shape[0]:
                                    medium[zz, yy, x] = True
    
    base_porosity = np.sum(medium) / medium.size
    print(f"Porosidad base de canales: {base_porosity:.3f}")
    
    # PASO 3: Colocar poros jerárquicamente
    print(f"\nColocando poros jerárquicamente...")
    
    # Coordenadas para esferas
    z_coords, y_coords, x_coords = np.mgrid[0:shape[0], 0:shape[1], 0:shape[2]]
    
    # Función auxiliar para colocar esferas
    def place_sphere(center, radius_um):
        radius_vox = radius_um / (2 * voxel_size_um)
        cz, cy, cx = center
        
        # Aplicar variación para heterogeneidad
        aspect_ratio = np.random.uniform(0.7, 1.3, size=3)
        
        sphere_mask = (
            ((z_coords - cz) / (radius_vox * aspect_ratio[0]))**2 + 
            ((y_coords - cy) / (radius_vox * aspect_ratio[1]))**2 + 
            ((x_coords - cx) / (radius_vox * aspect_ratio[2]))**2
        ) <= 1.0
        
        return sphere_mask
    
    # Colocar poros grandes
    print(f"  Colocando {n_large_pores} poros grandes...")
    margin_large = int(large_pore_size_um / (2 * voxel_size_um)) + 2
    
    for i in range(n_large_pores):
        # Distribución espacial más uniforme
        sector_x = (i % 3) * shape[2] // 3
        sector_y = ((i // 3) % 3) * shape[1] // 3
        sector_z = ((i // 9) % 3) * shape[0] // 3
        
        center = (
            np.random.randint(margin_large + sector_z, min(shape[0]-margin_large, sector_z + shape[0]//3)),
            np.random.randint(margin_large + sector_y, min(shape[1]-margin_large, sector_y + shape[1]//3)),
            np.random.randint(margin_large + sector_x, min(shape[2]-margin_large, sector_x + shape[2]//3))
        )
        
        # Variar tamaño para heterogeneidad
        size_variation = np.random.uniform(0.8, 1.2)
        medium |= place_sphere(center, large_pore_size_um * size_variation)
    
    # Colocar poros medianos
    print(f"  Colocando {n_medium_pores} poros medianos...")
    margin_medium = int(medium_pore_size_um / (2 * voxel_size_um)) + 1
    
    for i in range(n_medium_pores):
        for attempt in range(5):  # Múltiples intentos para buena distribución
            center = (
                np.random.randint(margin_medium, shape[0]-margin_medium),
                np.random.randint(margin_medium, shape[1]-margin_medium),
                np.random.randint(margin_medium, shape[2]-margin_medium)
            )
            
            # Preferir cerca de estructuras existentes
            if attempt > 2 or medium[center]:
                size_variation = np.random.uniform(0.7, 1.3)
                medium |= place_sphere(center, medium_pore_size_um * size_variation)
                break
    
    # Colocar poros pequeños
    print(f"  Colocando {n_small_pores} poros pequeños...")
    margin_small = int(small_pore_size_um / (2 * voxel_size_um)) + 1
    
    # Usar distance transform para colocar poros pequeños cerca de estructuras existentes
    dt = distance_transform_edt(~medium)
    
    for i in range(n_small_pores):
        # Buscar lugares cerca de poros existentes
        candidates = np.where((dt > 0) & (dt < 5))
        
        if len(candidates[0]) > 0:
            idx = np.random.randint(len(candidates[0]))
            center = (candidates[0][idx], candidates[1][idx], candidates[2][idx])
            
            # Verificar márgenes
            if (margin_small <= center[0] < shape[0]-margin_small and
                margin_small <= center[1] < shape[1]-margin_small and
                margin_small <= center[2] < shape[2]-margin_small):
                
                size_variation = np.random.uniform(0.6, 1.4)
                medium |= place_sphere(center, small_pore_size_um * size_variation)
    
    # PASO 4: Ajuste final de porosidad
    current_porosity = np.sum(medium) / medium.size
    print(f"\nPorosidad después de colocación: {current_porosity:.3f}")
    
    if abs(current_porosity - porosity_target) > 0.01:
        print(f"Ajustando porosidad...")
        
        if current_porosity > porosity_target:
            # Erosión suave
            iterations = 1 if current_porosity < porosity_target * 1.2 else 2
            struct = ndimage.generate_binary_structure(3, 1)
            medium = ndimage.binary_erosion(medium, structure=struct, iterations=iterations)
        elif current_porosity < porosity_target * 0.9:
            # Dilatación suave
            struct = ndimage.generate_binary_structure(3, 1)
            medium = ndimage.binary_dilation(medium, structure=struct, iterations=1)
    
    final_porosity = np.sum(medium) / medium.size
    
    # Verificar conectividad
    inlet = np.zeros(shape, dtype=bool)
    inlet[:, :, :5] = True
    connected = ps.filters.trim_disconnected_blobs(im=medium, inlets=inlet)
    connectivity = np.sum(connected) / np.sum(medium) if np.sum(medium) > 0 else 0
    
    print(f"\nResultados finales:")
    print(f"• Porosidad final: {final_porosity:.3f}")
    print(f"• Conectividad: {connectivity:.3f}")
    
    # Calcular heterogeneidad real
    dt_final = distance_transform_edt(medium)
    pore_radii = dt_final[medium] * voxel_size_um
    heterogeneity = np.std(pore_radii) / np.mean(pore_radii) if len(pore_radii) > 0 else 0
    print(f"• Coeficiente de heterogeneidad: {heterogeneity:.3f}")
    
    domain_size_um = [dim * voxel_size_um for dim in shape]
    
    return {
        'medium': medium,
        'shape': shape,
        'voxel_size_um': voxel_size_um,
        'domain_size_um': domain_size_um,
        'porosity_target': porosity_target,
        'porosity_actual': final_porosity,
        'connectivity': connectivity,
        'heterogeneity': heterogeneity,
        'pore_scales': {
            'large': large_pore_size_um,
            'medium': medium_pore_size_um,
            'small': small_pore_size_um
        }
    }


# Hierarchical Tree Network

In [ ]:
def create_heterogeneous_tight_rock(shape, porosity_target=0.10, voxel_size_um=0.05, seed=None):
    """
    Genera un medio poroso heterogéneo optimizado para rocas tight.
    
    MÉTODO ALTERNATIVO: Crecimiento desde múltiples semillas con conexión garantizada
    """
    
    print(f"="*60)
    print(f"GENERANDO MEDIO POROSO HETEROGÉNEO OPTIMIZADO")
    print(f"="*60)
    print(f"Dimensiones: {shape}")
    print(f"Porosidad objetivo: {porosity_target:.1%}")
    print(f"Tamaño de voxel: {voxel_size_um:.3f} μm")
    
    if seed is not None:
        np.random.seed(seed)
    
    # PASO 1: Definir distribución jerárquica de tamaños (mismas variables)
    large_pore_size_um = 0.8  
    n_large_pores = max(5, int(porosity_target * 0.1 * np.prod(shape) / 1000))
    
    medium_pore_size_um = 0.4  
    n_medium_pores = max(20, int(porosity_target * 0.3 * np.prod(shape) / 500))
    
    small_pore_size_um = 0.2  
    n_small_pores = max(100, int(porosity_target * 0.6 * np.prod(shape) / 200))
    
    print(f"\nDistribución jerárquica de poros:")
    print(f"• Grandes ({large_pore_size_um:.1f} μm): {n_large_pores} poros")
    print(f"• Medianos ({medium_pore_size_um:.1f} μm): {n_medium_pores} poros")
    print(f"• Pequeños ({small_pore_size_um:.1f} μm): {n_small_pores} poros")
    
    # Inicializar medio
    medium = np.zeros(shape, dtype=bool)
    
    # PASO 2: MÉTODO ALTERNATIVO - Crecimiento desde semillas conectadas
    print(f"\nCreando red conectada por crecimiento desde semillas...")
    
    # Crear semillas distribuidas que garanticen percolación
    n_seeds = 8  # Número de semillas principales
    seeds = []
    
    # Semillas en las caras del dominio para garantizar percolación
    # Cara izquierda (x=0)
    seeds.append((shape[0]//2, shape[1]//2, 0))
    # Cara derecha (x=max)
    seeds.append((shape[0]//2, shape[1]//2, shape[2]-1))
    # Caras intermedias para mejorar conectividad
    for i in range(1, 4):
        x_pos = int(i * shape[2] / 4)
        y_pos = np.random.randint(shape[1]//4, 3*shape[1]//4)
        z_pos = np.random.randint(shape[0]//4, 3*shape[0]//4)
        seeds.append((z_pos, y_pos, x_pos))
    
    # Crear árbol de expansión mínimo entre semillas
    from scipy.spatial import distance_matrix
    seed_positions = np.array(seeds)
    dist_matrix = distance_matrix(seed_positions, seed_positions)
    
    # Algoritmo de Prim para MST
    n = len(seeds)
    in_tree = [False] * n
    in_tree[0] = True
    edges = []
    
    for _ in range(n-1):
        min_edge = (None, None, float('inf'))
        for i in range(n):
            if in_tree[i]:
                for j in range(n):
                    if not in_tree[j] and dist_matrix[i,j] < min_edge[2]:
                        min_edge = (i, j, dist_matrix[i,j])
        
        if min_edge[0] is not None:
            edges.append((min_edge[0], min_edge[1]))
            in_tree[min_edge[1]] = True
    
    # Conectar semillas con caminos tortuosos
    channel_radius = int(small_pore_size_um / (2 * voxel_size_um))
    
    for edge in edges:
        start = seeds[edge[0]]
        end = seeds[edge[1]]
        
        # Crear camino tortuoso entre puntos
        n_points = int(np.linalg.norm(np.array(end) - np.array(start))) + 20
        
        # Añadir perturbaciones al camino
        t = np.linspace(0, 1, n_points)
        noise_amp = shape[0] // 10
        
        path_z = start[0] + t * (end[0] - start[0]) + noise_amp * np.sin(4*np.pi*t) * np.random.randn()
        path_y = start[1] + t * (end[1] - start[1]) + noise_amp * np.cos(3*np.pi*t) * np.random.randn()
        path_x = start[2] + t * (end[2] - start[2])
        
        # Dibujar el camino
        for i in range(len(t)):
            z, y, x = int(path_z[i]), int(path_y[i]), int(path_x[i])
            
            # Aplicar esfera en cada punto del camino
            for dz in range(-channel_radius, channel_radius+1):
                for dy in range(-channel_radius, channel_radius+1):
                    for dx in range(-channel_radius, channel_radius+1):
                        if dz**2 + dy**2 + dx**2 <= channel_radius**2:
                            zz, yy, xx = z+dz, y+dy, x+dx
                            if 0 <= zz < shape[0] and 0 <= yy < shape[1] and 0 <= xx < shape[2]:
                                medium[zz, yy, xx] = True
    
    # Añadir ramificaciones desde cada semilla
    for seed in seeds:
        n_branches = np.random.randint(3, 6)
        for _ in range(n_branches):
            # Dirección aleatoria de la rama
            direction = np.random.randn(3)
            direction = direction / np.linalg.norm(direction)
            
            # Longitud de la rama
            branch_length = np.random.randint(shape[0]//6, shape[0]//3)
            
            # Dibujar rama con grosor variable
            for t in range(branch_length):
                pos = seed + t * direction
                z, y, x = int(pos[0]), int(pos[1]), int(pos[2])
                
                # Radio decreciente
                current_radius = max(1, channel_radius - t//(branch_length//channel_radius))
                
                for dz in range(-current_radius, current_radius+1):
                    for dy in range(-current_radius, current_radius+1):
                        for dx in range(-current_radius, current_radius+1):
                            if dz**2 + dy**2 + dx**2 <= current_radius**2:
                                zz, yy, xx = z+dz, y+dy, x+dx
                                if 0 <= zz < shape[0] and 0 <= yy < shape[1] and 0 <= xx < shape[2]:
                                    medium[zz, yy, xx] = True
    
    base_porosity = np.sum(medium) / medium.size
    print(f"Porosidad base de red conectada: {base_porosity:.3f}")
    
    # PASO 3: Colocar poros jerárquicamente (con método de crecimiento preferencial)
    print(f"\nColocando poros jerárquicamente con crecimiento preferencial...")
    
    # Coordenadas para esferas
    z_coords, y_coords, x_coords = np.mgrid[0:shape[0], 0:shape[1], 0:shape[2]]
    
    # Función auxiliar para colocar esferas (igual que original)
    def place_sphere(center, radius_um):
        radius_vox = radius_um / (2 * voxel_size_um)
        cz, cy, cx = center
        
        aspect_ratio = np.random.uniform(0.7, 1.3, size=3)
        
        sphere_mask = (
            ((z_coords - cz) / (radius_vox * aspect_ratio[0]))**2 + 
            ((y_coords - cy) / (radius_vox * aspect_ratio[1]))**2 + 
            ((x_coords - cx) / (radius_vox * aspect_ratio[2]))**2
        ) <= 1.0
        
        return sphere_mask
    
    # Método alternativo: crecimiento preferencial cerca de estructuras existentes
    
    # Calcular mapa de probabilidad basado en distancia a poros existentes
    dt = distance_transform_edt(~medium)
    probability_map = np.exp(-dt / 5.0)  # Decaimiento exponencial
    probability_map[medium] = 0  # No colocar donde ya hay poros
    
    # Colocar poros grandes con distribución influenciada por la red
    print(f"  Colocando {n_large_pores} poros grandes...")
    margin_large = int(large_pore_size_um / (2 * voxel_size_um)) + 2
    
    for i in range(n_large_pores):
        # Seleccionar posición basada en probabilidad
        valid_mask = np.zeros_like(medium)
        valid_mask[margin_large:-margin_large, margin_large:-margin_large, margin_large:-margin_large] = 1
        
        prob = probability_map * valid_mask
        if np.sum(prob) > 0:
            prob = prob / np.sum(prob)
            flat_idx = np.random.choice(medium.size, p=prob.ravel())
            center = np.unravel_index(flat_idx, shape)
            
            size_variation = np.random.uniform(0.8, 1.2)
            medium |= place_sphere(center, large_pore_size_um * size_variation)
            
            # Actualizar mapa de probabilidad
            dt = distance_transform_edt(~medium)
            probability_map = np.exp(-dt / 5.0)
            probability_map[medium] = 0
    
    # Colocar poros medianos y pequeños con el mismo método
    print(f"  Colocando {n_medium_pores} poros medianos...")
    margin_medium = int(medium_pore_size_um / (2 * voxel_size_um)) + 1
    
    for i in range(n_medium_pores):
        valid_mask = np.zeros_like(medium)
        valid_mask[margin_medium:-margin_medium, margin_medium:-margin_medium, margin_medium:-margin_medium] = 1
        
        prob = probability_map * valid_mask
        if np.sum(prob) > 0:
            prob = prob / np.sum(prob)
            flat_idx = np.random.choice(medium.size, p=prob.ravel())
            center = np.unravel_index(flat_idx, shape)
            
            size_variation = np.random.uniform(0.7, 1.3)
            medium |= place_sphere(center, medium_pore_size_um * size_variation)
    
    print(f"  Colocando {n_small_pores} poros pequeños...")
    margin_small = int(small_pore_size_um / (2 * voxel_size_um)) + 1
    
    # Para poros pequeños, aumentar la influencia de proximidad
    dt = distance_transform_edt(~medium)
    probability_map = np.exp(-dt / 2.0)  # Decaimiento más rápido
    probability_map[medium] = 0
    
    for i in range(n_small_pores):
        valid_mask = np.zeros_like(medium)
        valid_mask[margin_small:-margin_small, margin_small:-margin_small, margin_small:-margin_small] = 1
        
        prob = probability_map * valid_mask
        if np.sum(prob) > 0:
            prob = prob / np.sum(prob)
            flat_idx = np.random.choice(medium.size, p=prob.ravel())
            center = np.unravel_index(flat_idx, shape)
            
            size_variation = np.random.uniform(0.6, 1.4)
            medium |= place_sphere(center, small_pore_size_um * size_variation)
    
    # PASO 4: Ajuste final de porosidad (igual que original)
    current_porosity = np.sum(medium) / medium.size
    print(f"\nPorosidad después de colocación: {current_porosity:.3f}")
    
    if abs(current_porosity - porosity_target) > 0.01:
        print(f"Ajustando porosidad...")
        
        if current_porosity > porosity_target:
            iterations = 1 if current_porosity < porosity_target * 1.2 else 2
            struct = ndimage.generate_binary_structure(3, 1)
            medium = ndimage.binary_erosion(medium, structure=struct, iterations=iterations)
        elif current_porosity < porosity_target * 0.9:
            struct = ndimage.generate_binary_structure(3, 1)
            medium = ndimage.binary_dilation(medium, structure=struct, iterations=1)
    
    final_porosity = np.sum(medium) / medium.size
    
    # Verificar conectividad
    inlet = np.zeros(shape, dtype=bool)
    inlet[:, :, :5] = True
    connected = ps.filters.trim_disconnected_blobs(im=medium, inlets=inlet)
    connectivity = np.sum(connected) / np.sum(medium) if np.sum(medium) > 0 else 0
    
    print(f"\nResultados finales:")
    print(f"• Porosidad final: {final_porosity:.3f}")
    print(f"• Conectividad: {connectivity:.3f}")
    
    # Calcular heterogeneidad real
    dt_final = distance_transform_edt(medium)
    pore_radii = dt_final[medium] * voxel_size_um
    heterogeneity = np.std(pore_radii) / np.mean(pore_radii) if len(pore_radii) > 0 else 0
    print(f"• Coeficiente de heterogeneidad: {heterogeneity:.3f}")
    
    domain_size_um = [dim * voxel_size_um for dim in shape]
    
    return {
        'medium': medium,
        'shape': shape,
        'voxel_size_um': voxel_size_um,
        'domain_size_um': domain_size_um,
        'porosity_target': porosity_target,
        'porosity_actual': final_porosity,
        'connectivity': connectivity,
        'heterogeneity': heterogeneity,
        'pore_scales': {
            'large': large_pore_size_um,
            'medium': medium_pore_size_um,
            'small': small_pore_size_um
        }
    }

# Degree-Constrained Network

In [ ]:
def create_heterogeneous_tight_rock(shape, porosity_target=0.10, voxel_size_um=0.05, seed=None):
    """
    Genera un medio poroso heterogéneo optimizado para rocas tight.
    
    MÉTODO DE RED CON NÚMEROS DE COORDINACIÓN: Genera poros como nodos conectados
    con números de coordinación específicos para garantizar percolación
    """
    
    print(f"="*60)
    print(f"GENERANDO MEDIO POROSO HETEROGÉNEO OPTIMIZADO")
    print(f"="*60)
    print(f"Dimensiones: {shape}")
    print(f"Porosidad objetivo: {porosity_target:.1%}")
    print(f"Tamaño de voxel: {voxel_size_um:.3f} μm")
    
    if seed is not None:
        np.random.seed(seed)
    
    # PASO 1: Definir distribución jerárquica de tamaños (mismas variables)
    large_pore_size_um = 0.8  
    n_large_pores = max(5, int(porosity_target * 0.1 * np.prod(shape) / 1000))
    
    medium_pore_size_um = 0.4  
    n_medium_pores = max(20, int(porosity_target * 0.3 * np.prod(shape) / 500))
    
    small_pore_size_um = 0.2  
    n_small_pores = max(100, int(porosity_target * 0.6 * np.prod(shape) / 200))
    
    print(f"\nDistribución jerárquica de poros:")
    print(f"• Grandes ({large_pore_size_um:.1f} μm): {n_large_pores} poros")
    print(f"• Medianos ({medium_pore_size_um:.1f} μm): {n_medium_pores} poros")
    print(f"• Pequeños ({small_pore_size_um:.1f} μm): {n_small_pores} poros")
    
    # Inicializar medio
    medium = np.zeros(shape, dtype=bool)
    
    # PASO 2: CREAR RED DE POROS CON NÚMEROS DE COORDINACIÓN
    print(f"\nCreando red de poros con números de coordinación controlados...")
    
    # Definir números de coordinación objetivo para cada tipo de poro
    # Poros grandes: alta conectividad (4-6 conexiones)
    # Poros medianos: conectividad media (3-4 conexiones)  
    # Poros pequeños: baja conectividad (2-3 conexiones)
    
    total_pores = n_large_pores + n_medium_pores + n_small_pores
    
    # Generar posiciones de nodos (centros de poros)
    pore_centers = []
    pore_sizes = []
    pore_types = []  # 0: pequeño, 1: mediano, 2: grande
    target_coordination = []
    
    # Función para generar posiciones con mínima distancia
    def generate_positions(n_pores, min_distance, pore_size, pore_type, coord_range):
        positions = []
        for _ in range(n_pores):
            attempts = 0
            while attempts < 100:
                pos = np.array([
                    np.random.randint(10, shape[0]-10),
                    np.random.randint(10, shape[1]-10),
                    np.random.randint(10, shape[2]-10)
                ])
                
                # Verificar distancia mínima con poros existentes
                valid = True
                for existing_pos in pore_centers:
                    if np.linalg.norm(pos - existing_pos) < min_distance:
                        valid = False
                        break
                
                if valid:
                    positions.append(pos)
                    pore_centers.append(pos)
                    pore_sizes.append(pore_size)
                    pore_types.append(pore_type)
                    target_coordination.append(np.random.randint(coord_range[0], coord_range[1]+1))
                    break
                
                attempts += 1
        
        return positions
    
    # Generar poros grandes primero (serán los nodos principales)
    print(f"  Generando {n_large_pores} nodos de poros grandes...")
    large_positions = generate_positions(
        n_large_pores, 
        large_pore_size_um / voxel_size_um * 2,
        large_pore_size_um,
        2,  # tipo grande
        (4, 6)  # coordinación 4-6
    )
    
    # Generar poros medianos
    print(f"  Generando {n_medium_pores} nodos de poros medianos...")
    medium_positions = generate_positions(
        n_medium_pores,
        medium_pore_size_um / voxel_size_um * 1.5,
        medium_pore_size_um,
        1,  # tipo mediano
        (3, 4)  # coordinación 3-4
    )
    
    # Generar poros pequeños
    print(f"  Generando {n_small_pores} nodos de poros pequeños...")
    small_positions = generate_positions(
        n_small_pores,
        small_pore_size_um / voxel_size_um * 1.2,
        small_pore_size_um,
        0,  # tipo pequeño
        (2, 3)  # coordinación 2-3
    )
    
    pore_centers = np.array(pore_centers)
    
    # PASO 3: CONECTAR POROS BASÁNDOSE EN NÚMEROS DE COORDINACIÓN
    print(f"\nConectando poros según números de coordinación...")
    
    # Calcular matriz de distancias
    from scipy.spatial import distance_matrix
    dist_matrix = distance_matrix(pore_centers, pore_centers)
    
    # Inicializar lista de conexiones
    connections = [[] for _ in range(len(pore_centers))]
    coordination_numbers = [0] * len(pore_centers)
    
    # Función para encontrar vecinos potenciales
    def find_potential_neighbors(idx, max_distance):
        distances = dist_matrix[idx]
        neighbors = []
        for j in range(len(pore_centers)):
            if j != idx and distances[j] < max_distance:
                neighbors.append((j, distances[j]))
        return sorted(neighbors, key=lambda x: x[1])
    
    # Conectar poros garantizando percolación
    # Primero, crear camino principal de percolación
    inlet_pores = []
    outlet_pores = []
    
    for i, center in enumerate(pore_centers):
        if center[2] < 10:  # Cerca del inlet (x=0)
            inlet_pores.append(i)
        elif center[2] > shape[2] - 10:  # Cerca del outlet (x=max)
            outlet_pores.append(i)
    
    # Asegurar al menos un camino de percolación
    if inlet_pores and outlet_pores:
        # Encontrar camino más corto entre inlet y outlet
        start_idx = inlet_pores[0]
        end_idx = outlet_pores[0]
        
        # Dijkstra simplificado para encontrar camino
        visited = [False] * len(pore_centers)
        distances = [float('inf')] * len(pore_centers)
        previous = [None] * len(pore_centers)
        distances[start_idx] = 0
        
        for _ in range(len(pore_centers)):
            # Encontrar nodo no visitado con menor distancia
            min_dist = float('inf')
            min_idx = -1
            for i in range(len(pore_centers)):
                if not visited[i] and distances[i] < min_dist:
                    min_dist = distances[i]
                    min_idx = i
            
            if min_idx == -1:
                break
                
            visited[min_idx] = True
            
            # Actualizar distancias de vecinos
            neighbors = find_potential_neighbors(min_idx, shape[0]/2)
            for neighbor_idx, dist in neighbors:
                if not visited[neighbor_idx]:
                    new_dist = distances[min_idx] + dist
                    if new_dist < distances[neighbor_idx]:
                        distances[neighbor_idx] = new_dist
                        previous[neighbor_idx] = min_idx
        
        # Reconstruir camino
        path = []
        current = end_idx
        while current is not None:
            path.append(current)
            current = previous[current]
        path.reverse()
        
        # Conectar nodos en el camino
        for i in range(len(path)-1):
            if path[i+1] not in connections[path[i]]:
                connections[path[i]].append(path[i+1])
                connections[path[i+1]].append(path[i])
                coordination_numbers[path[i]] += 1
                coordination_numbers[path[i+1]] += 1
    
    # Completar conexiones basándose en números de coordinación objetivo
    max_connection_distance = shape[0] / 3
    
    for i in range(len(pore_centers)):
        current_coordination = coordination_numbers[i]
        target_coord = target_coordination[i]
        
        if current_coordination < target_coord:
            neighbors = find_potential_neighbors(i, max_connection_distance)
            
            for neighbor_idx, _ in neighbors:
                if current_coordination >= target_coord:
                    break
                    
                neighbor_current = coordination_numbers[neighbor_idx]
                neighbor_target = target_coordination[neighbor_idx]
                
                # Conectar si ambos necesitan más conexiones
                if (neighbor_current < neighbor_target and 
                    neighbor_idx not in connections[i]):
                    
                    connections[i].append(neighbor_idx)
                    connections[neighbor_idx].append(i)
                    coordination_numbers[i] += 1
                    coordination_numbers[neighbor_idx] += 1
                    current_coordination += 1
    
    # PASO 4: CREAR GEOMETRÍA DE POROS Y GARGANTAS
    print(f"\nCreando geometría de poros y gargantas...")
    
    # Coordenadas para esferas
    z_coords, y_coords, x_coords = np.mgrid[0:shape[0], 0:shape[1], 0:shape[2]]
    
    # Función auxiliar para colocar esferas
    def place_sphere(center, radius_um):
        radius_vox = radius_um / (2 * voxel_size_um)
        cz, cy, cx = center
        
        aspect_ratio = np.random.uniform(0.7, 1.3, size=3)
        
        sphere_mask = (
            ((z_coords - cz) / (radius_vox * aspect_ratio[0]))**2 + 
            ((y_coords - cy) / (radius_vox * aspect_ratio[1]))**2 + 
            ((x_coords - cx) / (radius_vox * aspect_ratio[2]))**2
        ) <= 1.0
        
        return sphere_mask
    
    # Crear poros
    for i, center in enumerate(pore_centers):
        size_variation = np.random.uniform(0.8, 1.2)
        medium |= place_sphere(center, pore_sizes[i] * size_variation)
    
    # Crear gargantas (conexiones entre poros)
    channel_radius = int(small_pore_size_um / (2 * voxel_size_um))
    
    for i in range(len(pore_centers)):
        for j in connections[i]:
            if j > i:  # Evitar duplicados
                start = pore_centers[i]
                end = pore_centers[j]
                
                # Radio de garganta basado en los poros que conecta
                throat_radius = min(pore_sizes[i], pore_sizes[j]) * 0.3 / voxel_size_um
                throat_radius = max(1, int(throat_radius))
                
                # Crear cilindro conectando los poros
                n_points = int(np.linalg.norm(end - start)) + 10
                
                for t in np.linspace(0, 1, n_points):
                    point = start + t * (end - start)
                    z, y, x = int(point[0]), int(point[1]), int(point[2])
                    
                    # Crear sección circular
                    for dz in range(-throat_radius, throat_radius+1):
                        for dy in range(-throat_radius, throat_radius+1):
                            for dx in range(-throat_radius, throat_radius+1):
                                if dz**2 + dy**2 + dx**2 <= throat_radius**2:
                                    zz, yy, xx = z+dz, y+dy, x+dx
                                    if 0 <= zz < shape[0] and 0 <= yy < shape[1] and 0 <= xx < shape[2]:
                                        medium[zz, yy, xx] = True
    
    # Imprimir estadísticas de coordinación
    avg_coordination = np.mean(coordination_numbers)
    print(f"\nEstadísticas de números de coordinación:")
    print(f"• Coordinación promedio: {avg_coordination:.2f}")
    print(f"• Rango: {min(coordination_numbers)} - {max(coordination_numbers)}")
    
    base_porosity = np.sum(medium) / medium.size
    print(f"Porosidad base de red: {base_porosity:.3f}")
    
    # PASO 4: Ajuste final de porosidad (igual que original)
    current_porosity = np.sum(medium) / medium.size
    print(f"\nPorosidad después de colocación: {current_porosity:.3f}")
    
    if abs(current_porosity - porosity_target) > 0.01:
        print(f"Ajustando porosidad...")
        
        if current_porosity > porosity_target:
            iterations = 1 if current_porosity < porosity_target * 1.2 else 2
            struct = ndimage.generate_binary_structure(3, 1)
            medium = ndimage.binary_erosion(medium, structure=struct, iterations=iterations)
        elif current_porosity < porosity_target * 0.9:
            struct = ndimage.generate_binary_structure(3, 1)
            medium = ndimage.binary_dilation(medium, structure=struct, iterations=1)
    
    final_porosity = np.sum(medium) / medium.size
    
    # Verificar conectividad
    inlet = np.zeros(shape, dtype=bool)
    inlet[:, :, :5] = True
    connected = ps.filters.trim_disconnected_blobs(im=medium, inlets=inlet)
    connectivity = np.sum(connected) / np.sum(medium) if np.sum(medium) > 0 else 0
    
    print(f"\nResultados finales:")
    print(f"• Porosidad final: {final_porosity:.3f}")
    print(f"• Conectividad: {connectivity:.3f}")
    
    # Calcular heterogeneidad real
    dt_final = distance_transform_edt(medium)
    pore_radii = dt_final[medium] * voxel_size_um
    heterogeneity = np.std(pore_radii) / np.mean(pore_radii) if len(pore_radii) > 0 else 0
    print(f"• Coeficiente de heterogeneidad: {heterogeneity:.3f}")
    
    domain_size_um = [dim * voxel_size_um for dim in shape]
    
    return {
        'medium': medium,
        'shape': shape,
        'voxel_size_um': voxel_size_um,
        'domain_size_um': domain_size_um,
        'porosity_target': porosity_target,
        'porosity_actual': final_porosity,
        'connectivity': connectivity,
        'heterogeneity': heterogeneity,
        'pore_scales': {
            'large': large_pore_size_um,
            'medium': medium_pore_size_um,
            'small': small_pore_size_um
        }
    }